# LoRA fine-tuning on CPU in a few minutes

The whole SageMaker fine-tuning loop, end to end, on CPU instances only, so it runs in any account without
waiting for GPU capacity:

1. a training job fine-tunes **Qwen3-0.6B** with a LoRA adapter (about 20 steps) and evaluates the base and the
   fine-tuned model on held-out examples, in the same job. It is launched three times in parallel to compare the
   ways of getting capacity: **on-demand**, **Spot**, and an instance-type **fallback** list
2. you read the metrics and see what changed
3. the result is deployed three ways: a **real-time** endpoint, a real-time endpoint with **instance pools**
   (fallback), and a **serverless** endpoint

**The task.** Turn a free-text restaurant booking request into strict JSON:

`"hey! Dana - four ppl friday 7pm if possible"` → `{"name": "Dana", "party_size": 4, "day": "Friday", "time": "19:00"}`

The data is generated in code (`scripts/booking_data.py`), so nothing is downloaded except the model. The base
model gets the same instruction, and usually gets the values roughly right but not the exact format (it wraps
the JSON in a markdown code block, keeps "friday" lowercase, writes "7pm"). A few LoRA steps teach the format.

The model is small and the run short on purpose: this notebook shows the mechanics (training job, adapter,
metrics, endpoint). The same pattern scales to GPU instances and larger models in the other labs.

In [ ]:
%pip install -q "sagemaker==3.21.0"

In [ ]:
import json
import time

import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
region = sess.boto_region_name
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]
bucket = sess.default_bucket()
prefix = "/".join(p for p in (sess.default_bucket_prefix, "cpu-fine-tuning-techniques") if p)
sm = boto3.client("sagemaker", region_name=region)
s3 = boto3.client("s3", region_name=region)
print(f"role:   {role}\nregion: {region}\noutput: s3://{bucket}/{prefix}/")

## The data

In [ ]:
import sys

sys.path.insert(0, "scripts")
from booking_data import INSTRUCTION, make_splits

train, test = make_splits(n_train=400, n_test=30)
print(f"instruction: {INSTRUCTION}\n{len(train)} training / {len(test)} test examples\n")
for ex in train[:3]:
    print(f"{ex['request']}\n  -> {ex['target']}")

## Train and evaluate: three ways to get capacity

One training job does everything (`scripts/train.py`): it scores the base model, trains the adapter, scores the
fine-tuned model, and writes `adapter/` and `metrics.json`. The same job is launched three times, in parallel,
each asking SageMaker for capacity in a different way:

| `capacity` | What SageMaker does | When to use it |
|---|---|---|
| `on-demand` | runs on exactly `INSTANCE_TYPE` | the default; predictable |
| `spot` | runs on `INSTANCE_TYPE` using spare (Spot) capacity, waiting up to `max_wait` for it; can be interrupted | cheaper; for jobs that can wait, or resume from checkpoints |
| `fallback` | runs on the first type in `FALLBACK_TYPES` that has capacity (`instance_preferences`) | when one type is often unavailable (typical for GPUs) |

SageMaker rejects a fallback list if any type in it has no training quota, so types without quota are dropped
first. Outputs are stored uncompressed so the endpoints below can use them directly.

In [ ]:
from sagemaker.core.shapes import InstancePreference
from sagemaker.core.training.configs import Compute, OutputDataConfig, SourceCode, StoppingCondition
from sagemaker.train import ModelTrainer

STEPS = 20
INSTANCE_TYPE = "ml.c7i.4xlarge"
FALLBACK_TYPES = ["ml.c7i.4xlarge", "ml.m7i.4xlarge", "ml.c6i.4xlarge", "ml.m6i.4xlarge", "ml.m5.4xlarge"]

try:
    quotas = {q["QuotaName"]: q["Value"]
              for page in boto3.client("service-quotas", region_name=region).get_paginator("list_service_quotas")
              .paginate(ServiceCode="sagemaker") for q in page["Quotas"]}
    FALLBACK_TYPES = [t for t in FALLBACK_TYPES if quotas.get(f"{t} for training job usage", 0) >= 1] or FALLBACK_TYPES
except Exception as e:  # role cannot read Service Quotas: send the full list
    print(f"quota check skipped: {e}")


def make_trainer(capacity):
    if capacity == "fallback":
        compute = Compute(instance_preferences=[InstancePreference(instance_type=t) for t in FALLBACK_TYPES],
                          instance_count=1, volume_size_in_gb=30)
    else:
        compute = Compute(instance_type=INSTANCE_TYPE, instance_count=1, volume_size_in_gb=30,
                          enable_managed_spot_training=capacity == "spot")
    return ModelTrainer(
        training_image=f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-training:2.8-cpu-py312",
        source_code=SourceCode(source_dir="scripts", requirements="requirements.txt", entry_script="train.py"),
        hyperparameters={"steps": STEPS, "model_id": "Qwen/Qwen3-0.6B"},
        compute=compute,
        # Spot only: max_wait is the total time allowed, including waiting for Spot capacity; past it the job
        # stops instead of waiting for capacity indefinitely
        stopping_condition=StoppingCondition(max_runtime_in_seconds=1200,
                                             max_wait_time_in_seconds=1800 if capacity == "spot" else None),
        output_data_config=OutputDataConfig(s3_output_path=f"s3://{bucket}/{prefix}/jobs", compression_type="NONE"),
        base_job_name=f"cpu-ft-sft-{capacity}",
        role=role,
        sagemaker_session=sess,
    )


jobs = {}
for capacity in ("on-demand", "spot", "fallback"):
    trainer = make_trainer(capacity)
    trainer.train(wait=False)
    jobs[capacity] = trainer._latest_training_job.training_job_name
    print(f"{capacity:10} submitted: {jobs[capacity]}")
print(f"fallback list: {FALLBACK_TYPES}")

### Wait for the three jobs

The jobs run in SageMaker independently of this notebook. If the kernel restarts, re-run the setup cell and this
one: it finds the latest job of each kind.

In [ ]:
jobs = globals().get("jobs") or {
    c: sm.list_training_jobs(NameContains=f"cpu-ft-sft-{c}", SortBy="CreationTime", SortOrder="Descending",
                             MaxResults=1)["TrainingJobSummaries"][0]["TrainingJobName"]
    for c in ("on-demand", "spot", "fallback")}
GRACE_S = 600  # once on-demand is done, wait at most this long for the others (e.g. Spot without capacity)
last, done_at = {}, None
while True:
    desc = {c: sm.describe_training_job(TrainingJobName=n) for c, n in jobs.items()}
    for c, d in desc.items():
        line = f"{d['TrainingJobStatus']}/{d['SecondaryStatus']}"
        if last.get(c) != line:
            print(time.strftime("%H:%M:%S"), f"{c:10}", line, flush=True)
            last[c] = line
    finished = {c for c, d in desc.items() if d["TrainingJobStatus"] in ("Completed", "Failed", "Stopped")}
    if len(finished) == len(desc):
        break
    if "on-demand" in finished:
        done_at = done_at or time.time()
        if time.time() - done_at > GRACE_S:
            print(f"\nstill running: {sorted(set(desc) - finished)}. Continuing with the on-demand model; "
                  "re-run this cell later to see how they ended.")
            break
    time.sleep(15)


def phases(d):
    out = {}
    for t in d.get("SecondaryStatusTransitions", []):
        end = t.get("EndTime") or d.get("TrainingEndTime")
        if end:
            out[t["Status"]] = (end - t["StartTime"]).total_seconds()
    return out


print(f"\n{'capacity':10} {'status':10} {'instance':16} {'to start':>8} {'training':>8} {'billed':>7} {'spot savings':>12}")
for c, d in desc.items():
    rc, p = d["ResourceConfig"], phases(d)
    savings = ""
    if c == "spot" and d.get("TrainingTimeInSeconds"):
        savings = f"{1 - d['BillableTimeInSeconds'] / d['TrainingTimeInSeconds']:.0%}"
    print(f"{c:10} {d['TrainingJobStatus']:10} {rc.get('SelectedInstanceType') or rc.get('InstanceType'):16} "
          f"{p.get('Starting', 0) + p.get('Downloading', 0):7.0f}s {p.get('Training', 0):7.0f}s "
          f"{d.get('BillableTimeInSeconds', 0):6}s {savings:>12}")

The three jobs train the same model on the same data, so their results match; only the capacity path differs.
"spot savings" is what SageMaker reports for the job (`1 - billable time / training time`).

## Results

In [ ]:
def metrics_of(job_name):
    d = sm.describe_training_job(TrainingJobName=job_name)
    prefix_uri = d["ModelArtifacts"]["S3ModelArtifacts"].rstrip("/") + "/"
    b, k = prefix_uri[5:].split("/", 1)
    return prefix_uri, json.loads(s3.get_object(Bucket=b, Key=k + "metrics.json")["Body"].read())


model_prefix, metrics = metrics_of(jobs["on-demand"])
print(f"{'':10} {'exact':>6} " + " ".join(f"{f:>10}" for f in ("name", "party_size", "day", "time")))
for which in ("base", "finetuned"):
    m = metrics[which]
    print(f"{which:10} {m['exact_match']:6.0%} " + " ".join(f"{m['acc_' + f]:10.0%}" for f in ("name", "party_size", "day", "time")))
print(f"\n{metrics['steps']} steps, {metrics['seconds_training']} s training\n")
for ex in metrics["examples"][:2]:
    print(f"request:   {ex['request']}\nbase:      {ex['base']!r}\nfinetuned: {ex['finetuned']}\n")
for c in ("spot", "fallback"):
    if desc[c]["TrainingJobStatus"] == "Completed":
        print(f"{c:10} fine-tuned exact match: {metrics_of(jobs[c])[1]['finetuned']['exact_match']:.0%}")

## Serve it: three ways to deploy

All three endpoints run the same model (base model from the Hugging Face Hub + the adapter from the on-demand job,
merged at startup by `serving/inference.py`) and differ only in how capacity is provided:

| endpoint | What SageMaker does | Billing |
|---|---|---|
| `realtime` | one `ml.m5.xlarge` instance, always on | per instance-hour |
| `realtime-pools` | the first type in a priority list that has capacity (`InstancePools`, up to 5) | per instance-hour |
| `serverless` | starts compute on demand; the first request after idle time waits for a cold start | per request (compute time) |

They take a few minutes to create. The real-time ones bill per hour until deleted (last cell).

In [ ]:
import tarfile
import tempfile
from pathlib import Path

# Package the adapter (~9 MB) + the handler code as model.tar.gz: serverless endpoints need a tarball
# (ModelDataUrl); the base model is downloaded from the Hub when the container starts.
mb, mk = model_prefix[5:].split("/", 1)
work = Path(tempfile.mkdtemp())
for obj in s3.list_objects_v2(Bucket=mb, Prefix=mk + "adapter/")["Contents"]:
    dest = work / "adapter" / obj["Key"].rsplit("/", 1)[-1]
    dest.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(mb, obj["Key"], str(dest))
with tarfile.open(work / "model.tar.gz", "w:gz") as tar:
    tar.add(work / "adapter", arcname="adapter")
    for name in ("inference.py", "requirements.txt"):
        tar.add(f"serving/{name}", arcname=f"code/{name}")
s3.upload_file(str(work / "model.tar.gz"), mb, mk + "serving/model.tar.gz")

container = {
    "Image": f"763104351884.dkr.ecr.{region}.amazonaws.com/pytorch-inference:2.6-cpu-py312",
    "ModelDataUrl": f"s3://{mb}/{mk}serving/model.tar.gz",
    "Environment": {"SAGEMAKER_PROGRAM": "inference.py", "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
                    "SAGEMAKER_MODEL_SERVER_WORKERS": "1"},
}
variants = {
    "realtime": {"InstanceType": "ml.m5.xlarge", "InitialInstanceCount": 1},
    "realtime-pools": {"InstancePools": [{"InstanceType": t, "Priority": i + 1}
                                         for i, t in enumerate(["ml.m5.xlarge", "ml.m7i.xlarge", "ml.c7i.2xlarge"])],
                       "InitialInstanceCount": 1, "VariantInstanceProvisionTimeoutInSeconds": 1800},
    "serverless": {"ServerlessConfig": {"MemorySizeInMB": 6144, "MaxConcurrency": 1}},
}
stamp = time.strftime("%Y%m%d-%H%M%S")
endpoints = {}
for kind, variant in variants.items():
    name = f"cpu-ft-sft-{kind}-{stamp}"
    sm.create_model(ModelName=name, ExecutionRoleArn=role, PrimaryContainer=container)
    sm.create_endpoint_config(EndpointConfigName=name,
                              ProductionVariants=[{"VariantName": "AllTraffic", "ModelName": name, **variant}])
    sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
    endpoints[kind] = name
t0 = time.time()
for kind, name in endpoints.items():
    try:
        sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 20, "MaxAttempts": 90})
        print(f"{kind:15} in service after {time.time() - t0:.0f}s")
    except Exception:
        print(f"{kind:15} {sm.describe_endpoint(EndpointName=name).get('FailureReason')}")

In [ ]:
from botocore.config import Config

runtime = boto3.client("sagemaker-runtime", region_name=region,
                       config=Config(read_timeout=120, retries={"max_attempts": 2}))
request = "hey! Dana - four ppl friday 7pm if possible"
for kind, name in endpoints.items():
    for attempt in ("first", "second"):
        t = time.time()
        try:
            resp = runtime.invoke_endpoint(EndpointName=name, ContentType="application/json",
                                           Body=json.dumps({"request": request}))
            out = json.loads(resp["Body"].read())["booking"]
        except Exception as e:
            out = f"error: {str(e)[:120]}"
        print(f"{kind:15} {attempt:6} {time.time() - t:5.1f}s  {out}")

The serverless endpoint's first request after idle time includes the cold start; the second one is warm.

In [ ]:
# Clean up the endpoints (the real-time ones bill per hour while they exist)
for name in endpoints.values():
    sm.delete_endpoint(EndpointName=name)
    sm.delete_endpoint_config(EndpointConfigName=name)
    sm.delete_model(ModelName=name)
    print("deleted", name)